# 🚩 フラグ実在（parse_args 判定）

`--help` の grep ではなく argparse に通して判定する。
cfgresolve は import で落ちてここに到達しなかった。


In [ ]:
import subprocess
import sys

print("=== pip install vllm==0.32.0 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "vllm==0.32.0"], capture_output=True, text=True, timeout=5400)
tail = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
print("   rc:", r.returncode, "|", " / ".join(tail[-3:])[:300])
r = subprocess.run([sys.executable, "-c",
                    "import vllm, torch, transformers;"
                    "print(vllm.__version__, torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("   versions:", (r.stdout or r.stderr).strip()[:300])
def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

ROOT = find_root()
TPL = ROOT + "/chat_template.jinja"
import os
print("ROOT:", ROOT, "| template:", os.path.isfile(TPL))

CASE = r"""
import json
import sys
import traceback

ROOT, TPL = sys.argv[1], sys.argv[2]

from vllm.entrypoints.launchers.cli_args import (
    make_arg_parser, validate_parsed_serve_args)
from vllm.utils.argparse_utils import FlexibleArgumentParser

p = make_arg_parser(FlexibleArgumentParser())
SPEC_JSON = json.dumps({"method": "mtp", "num_speculative_tokens": 2})

# builder が実際に出すフラグを全部。値は本番と同じ。
FLAGS = [
    ("--host", ["127.0.0.1"]),
    ("--port", ["8000"]),
    ("--served-model-name", ["qwen3.8-27b-awq"]),
    ("--tensor-parallel-size", ["2"]),
    ("--dtype", ["half"]),
    ("--quantization", ["compressed-tensors"]),
    ("--max-model-len", ["32768"]),
    ("--max-num-seqs", ["4"]),
    ("--max-num-batched-tokens", ["4096"]),
    ("--gpu-memory-utilization", ["0.9"]),
    ("--kv-cache-dtype", ["auto"]),
    ("--swap-space", ["0"]),
    ("--cpu-offload-gb", ["0"]),
    ("--trust-remote-code", []),
    ("--reasoning-parser", ["qwen3"]),
    ("--enforce-eager", []),
    ("--enable-prefix-caching", []),
    ("--enable-auto-tool-choice", []),
    ("--tool-call-parser", ["qwen3_xml"]),
    ("--speculative-config", [SPEC_JSON]),
    ("--spec-method", ["mtp"]),
    ("--language-model-only", []),
    ("--limit-mm-per-prompt", ['{"image":0,"video":0}']),
    ("--chat-template", [TPL]),
    ("--chat-template-content-format", ["string"]),
    ("--default-chat-template-kwargs", ['{"reasoning_effort":"medium"}']),
    ("--trust-request-chat-template", []),
    ("--disable-log-requests", []),
    ("--enable-log-requests", []),
    ("--enable-chunked-prefill", []),
    ("--compilation-config", ['{"level":0}']),
    ("--load-format", ["safetensors"]),
    ("--mm-processor-kwargs", ["{}"]),
    ("--disable-mm-preprocessor-cache", []),
    ("--uvicorn-log-level", ["warning"]),
    ("--api-key", ["k"]),
    ("--allowed-origins", ['["*"]']),
    ("--tokenizer", [ROOT]),
    ("--seed", ["0"]),
    ("--distributed-executor-backend", ["mp"]),
    ("--disable-custom-all-reduce", []),
    ("--generation-config", ["auto"]),
    ("--override-generation-config", ['{"repetition_penalty":1.05}']),
    ("--attention-backend", ["TRITON_ATTN"]),
    ("--enforce-sampling", []),
    ("--fail-on-environ-validation", []),
    ("--served-model-name-extra", ["x"]),
]

ok, ng = [], []
for flag, vals in FLAGS:
    try:
        p.parse_args([ROOT] + [flag] + vals)
        ok.append(flag)
        print(f"  [OK] {flag}")
    except SystemExit as e:
        ng.append(flag)
        print(f"  [NG] {flag} (SystemExit {e.code})")
    except Exception as e:
        ng.append(flag)
        print(f"  [NG] {flag} ({type(e).__name__}: {str(e)[:160]})")

print("\n--- まとめ ---")
print("OK:", len(ok), "NG:", len(ng))
print("OK_FLAGS =", json.dumps(ok, ensure_ascii=False))
print("NG_FLAGS =", json.dumps(ng, ensure_ascii=False))

print("\n=== builder が実際に出す組み合わせ 1 回の parse ===")
ALL = [ROOT,
       "--served-model-name", "qwen3.8-27b-awq",
       "--host", "127.0.0.1", "--port", "8000",
       "--tensor-parallel-size", "2", "--dtype", "half",
       "--max-num-seqs", "4", "--gpu-memory-utilization", "0.9",
       "--kv-cache-dtype", "auto",
       "--trust-remote-code", "--reasoning-parser", "qwen3",
       "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_xml",
       "--language-model-only",
       "--speculative-config", SPEC_JSON,
       "--chat-template", TPL,
       "--max-model-len", "32768"]
try:
    ns = p.parse_args(ALL)
    print("[OK] parse(ALL)")
except SystemExit as e:
    print("[NG] parse(ALL) SystemExit", e.code)
    raise SystemExit(0)
except Exception as e:
    print("[NG] parse(ALL)", type(e).__name__, str(e)[:400])
    raise SystemExit(0)

for k in ("language_model_only", "speculative_config", "reasoning_parser",
          "tool_call_parser", "chat_template", "max_model_len",
          "kv_cache_dtype", "quantization", "dtype", "trust_remote_code"):
    print(f"    {k} = {getattr(ns, k, '<missing>')!r}")
try:
    validate_parsed_serve_args(ns)
    print("[OK] validate_parsed_serve_args")
except Exception as e:
    print("[NG] validate_parsed_serve_args:", type(e).__name__, str(e)[:400])

print("\n=== chat template の validate ===")
from vllm.renderers.chat_utils import validate_chat_template
for tag, arg in (("file", TPL), ("nonexistent", "/nope.jinja"), ("None", None)):
    try:
        validate_chat_template(arg)
        print(f"  [OK] {tag}")
    except Exception as e:
        print(f"  [NG] {tag}: {type(e).__name__}: {str(e)[:200]}")

print("\n=== どのモジュールに get_tokenizer が居るか ===")
for mod in ("vllm.tokenizers", "vllm.transformers_utils.tokenizer",
            "vllm.transformers_utils.tokenizer_group"):
    try:
        m = __import__(mod, fromlist=["x"])
        print(f"  [OK] {mod} ->", hasattr(m, "get_tokenizer"))
    except Exception as e:
        print(f"  [NG] {mod}: {type(e).__name__}: {str(e)[:120]}")
"""

import os
import subprocess
import sys
import tempfile

case = os.path.join(tempfile.gettempdir(), "flagcase.py")
open(case, "w").write(CASE)
r = subprocess.run([sys.executable, case, ROOT, TPL],
                   capture_output=True, text=True, timeout=3600)
print((r.stdout or "")[-16000:])
if r.returncode != 0:
    print("=== stderr ===")
    print((r.stderr or "")[-4000:])
